# Player Age Data
This workbook contains a script to download player dates of birth from Wikipedia. The script takes about an hour to run.

In [4]:
import pandas as pd
import requests
import re
import time

HEADERS = {
    "User-Agent": "FootballAgeResearch/1.0 (student project; derrickbauer@gmail.com)"
}

def get_birth_year_from_wikipedia(player_name):
    # Search Wikipedia for a player and extract their birth year from the infobox or article text.
    
    # Step 1: Search for the player
    search_url = "https://en.wikipedia.org/w/api.php"
    search_params = {
        "action":   "query",
        "list":     "search",
        "srsearch": f"{player_name} footballer",
        "format":   "json",
        "srlimit":  1
    }

    try:
        r = requests.get(search_url, params=search_params, timeout=10, headers=HEADERS)
        results = r.json().get("query", {}).get("search", [])
        if not results:
            return None

        page_title = results[0]["title"]

        # Step 2: Fetch the page content
        content_params = {
            "action":      "query",
            "titles":      page_title,
            "prop":        "revisions",
            "rvprop":      "content",
            "format":      "json",
            "rvslots":     "main",
            "redirects":   1
        }

        r2 = requests.get(search_url, params=content_params, timeout=10, headers=HEADERS)
        pages = r2.json()["query"]["pages"]
        page = next(iter(pages.values()))
        content = page["revisions"][0]["slots"]["main"]["*"]

        # Step 3: Extract birth date from infobox
        # Looks for {{birth date|YYYY|MM|DD}} or |birth_date = 
        patterns = [
            r"\{\{birth date(?:\|df=yes)?\|(\d{4})\|",   # {{birth date|YYYY|...
            r"\{\{birth date and age\|(\d{4})\|",          # {{birth date and age|YYYY|...
            r"\|\s*birth_date\s*=\s*.*?(\d{4})",           # |birth_date = ... YYYY
            r"born\s+\w+\s+\d+,\s+(\d{4})",               # born Month DD, YYYY
            r"born.*?(\d{4})",                              # fallback: born ... YYYY
        ]

        for pattern in patterns:
            match = re.search(pattern, content, re.IGNORECASE)
            if match:
                year = int(match.group(1))
                if 1960 <= year <= 2010:  # sanity check
                    return year

        return None

    except Exception as e:
        print(f"Error fetching {player_name}: {e}")
        return None


def enrich_with_birth_year(df, name_col='name', delay=0.5):
    # Add birth_year column to a dataframe by looking up each player on Wikipedia.
    
    birth_years = []
    total = len(df)

    print(f"Looking up birth years for {total} players.")

    for i, name in enumerate(df[name_col]):
        print(f"[{i+1}/{total}] Looking up: {name}")
        year = get_birth_year_from_wikipedia(name)
        birth_years.append(year)
        time.sleep(delay)  # be polite to Wikipedia's servers

    df['birth_year'] = birth_years
    
    matched = df['birth_year'].notna().sum()
    print(f"\nDone. Found birth years for {matched}/{total} players ({matched/total*100:.1f}%)")
    return df


In [8]:
df_profiles = pd.read_csv('data/all_player_profiles.csv')

df_profiles_with_birth_year = enrich_with_birth_year(df_profiles, name_col='name')

print(df_profiles_with_birth_year[['name', 'birth_year']].head(20))
print(f"\nMissing birth year: {df_profiles_with_birth_year['birth_year'].isna().sum()}")

Looking up birth years for 4682 players.
[1/4682] Looking up: Viktor Gyökeres
[2/4682] Looking up: Bukayo Saka
[3/4682] Looking up: Gabriel Jesus
[4/4682] Looking up: Gabriel Martinelli
[5/4682] Looking up: Kai Havertz
[6/4682] Looking up: Leandro Trossard
[7/4682] Looking up: Brando Bailey-Joseph
[8/4682] Looking up: Martin Ødegaard
[9/4682] Looking up: Declan Rice
[10/4682] Looking up: Eberechi Eze
[11/4682] Looking up: Noni Madueke
[12/4682] Looking up: Mikel Merino
[13/4682] Looking up: Max Dowman
[14/4682] Looking up: Martín Zubimendi
[15/4682] Looking up: Christian Nørgaard
[16/4682] Looking up: Andre Harriman-Annous
[17/4682] Looking up: Ife Ibrahim
[18/4682] Looking up: Ceadach O'Neill
[19/4682] Looking up: Gabriel Magalhães
[20/4682] Looking up: William Saliba
[21/4682] Looking up: Piero Hincapié
[22/4682] Looking up: Riccardo Calafiori
[23/4682] Looking up: Jurriën Timber
[24/4682] Looking up: Myles Lewis-Skelly
[25/4682] Looking up: Cristhian Mosquera
[26/4682] Looking up: B

In [7]:
df_profiles_with_birth_year[['player_id', 'name', 'birth_year']].to_csv('data/player_birth_years.csv', index=False)